# Train YOLO Models

## 1. Gather and Label Data

Before we start training, we need to gather and label images that will be used for training the object detection model. I built a custom dataset by taking 15 pictures of 3 objects and labeling them using [Label Studio](https://labelstud.io/)

<p align=center>
<video width="640" height="360" controls>
  <source src="../screenshots/data_labelling.webm" type="video/webm" width="800">
  Your browser does not support the video tag.
</video>
<br>
<i>Data Labeling in Label Studio.</i>
</p>

The images were exported in a `data.zip` file that contains the following:

- An `images` folder containing the images
- A `labels` folder containing the labels in YOLO annotation format
- A `classes.txt` labelmap file that contains all the classes
- A `notes.json` file that contains info specific to Label Studio (this file can be ignored)

<p align=center>
<img src="../screenshots/data_structure.png" width="800"><br>
<i>Data structure</i>
</p>

## 2. Train/Val split:

In [1]:
# importing modules
from pathlib import Path
import random
import os
import shutil

In [2]:
# train/val split
train_percent = 0.8
val_percent = 1 - train_percent

In [3]:
# define path to input dataset 
cwd = os.getcwd()
input_image_path = os.path.join(cwd,'../data/images')
input_label_path = os.path.join(cwd,'../data/labels')

train_img_path = os.path.join(cwd,'../data/train/images')
train_txt_path = os.path.join(cwd,'../data/train/labels')
val_img_path = os.path.join(cwd,'../data/validation/images')
val_txt_path = os.path.join(cwd,'../data/validation/labels')

In [4]:
# Create folders if they don't already exist
for dir_path in [train_img_path, train_txt_path, val_img_path, val_txt_path]:
   if not os.path.exists(dir_path):
      os.makedirs(dir_path)
      print(f'Created folder at {dir_path}.')

Created folder at /home/ritisha/Projects/train-yolo-model/notebooks/../data/train/images.
Created folder at /home/ritisha/Projects/train-yolo-model/notebooks/../data/train/labels.
Created folder at /home/ritisha/Projects/train-yolo-model/notebooks/../data/validation/images.
Created folder at /home/ritisha/Projects/train-yolo-model/notebooks/../data/validation/labels.


In [5]:
# Get list of all images and annotation files
img_file_list = [path for path in Path(input_image_path).rglob('*')]
txt_file_list = [path for path in Path(input_label_path).rglob('*')]

print(f'Number of image files: {len(img_file_list)}')
print(f'Number of annotation files: {len(txt_file_list)}')

Number of image files: 45
Number of annotation files: 45


In [6]:
# Determine number of files to move to each folder
file_num = len(img_file_list)
train_num = int(file_num*train_percent)
val_num = file_num - train_num
print('Images moving to train: %d' % train_num)
print('Images moving to validation: %d' % val_num)

Images moving to train: 36
Images moving to validation: 9


In [7]:
# Select files randomly and copy them to train or val folders
for i, set_num in enumerate([train_num, val_num]):
  for ii in range(set_num):
    img_path = random.choice(img_file_list)
    img_fn = img_path.name
    base_fn = img_path.stem
    txt_fn = base_fn + '.txt'
    txt_path = os.path.join(input_label_path,txt_fn)

    if i == 0: # Copy first set of files to train folders
      new_img_path, new_txt_path = train_img_path, train_txt_path
    elif i == 1: # Copy second set of files to the validation folders
      new_img_path, new_txt_path = val_img_path, val_txt_path

    shutil.copy(img_path, os.path.join(new_img_path,img_fn))
    #os.rename(img_path, os.path.join(new_img_path,img_fn))
    if os.path.exists(txt_path): # If txt path does not exist, this is a background image, so skip txt file
      shutil.copy(txt_path,os.path.join(new_txt_path,txt_fn))
      #os.rename(txt_path,os.path.join(new_txt_path,txt_fn))

    img_file_list.remove(img_path)

## 3. Configure Training

In [ ]:
import yaml
import os

cwd = os.getcwd()
path_to_classes_txt = os.path.join(cwd, '../data/classes.txt')
path_to_data_yaml = os.path.abspath(os.path.join(cwd, '../data.yaml'))
data_path = os.path.abspath(os.path.join(cwd, '../data'))

# Read class names from classes.txt
with open(path_to_classes_txt, 'r') as f:
    classes = [line.strip() for line in f if line.strip()]

# Write data.yaml
data = {
    'path': data_path,
    'train': 'train/images',
    'val': 'validation/images',
    'nc': len(classes),
    'names': classes
}
with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)

print(f'Created data.yaml at: {path_to_data_yaml}')
print(f'Classes ({len(classes)}): {classes}')


## 4. Install Requirements

In [ ]:
!pip install ultralytics

## 5. Train Model

**Model size options** (trade speed vs accuracy):
- `yolo11n.pt` — nano (fastest, least accurate)
- `yolo11s.pt` — small (good starting point)
- `yolo11m.pt` — medium
- `yolo11l.pt` — large (slowest, most accurate)

**Epochs:** Since our dataset has <200 images, 60 epochs is a good starting point.

**imgsz:** 640 is the standard resolution for YOLO models.

In [ ]:
from ultralytics import YOLO

# Load a pretrained model to fine-tune
model = YOLO('yolo11s.pt')

# Train on our dataset
results = model.train(
    data=path_to_data_yaml,
    epochs=60,
    imgsz=640
)

print('Training complete!')
print(f'Best model saved at: {results.save_dir}/weights/best.pt')


## 6. Test Model

Run the trained model on the validation images and display the results.

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Build absolute paths so they work regardless of working directory
project_root = os.path.abspath(os.path.join(cwd, '..'))
best_model_path = os.path.join(project_root, 'runs/detect/train/weights/best.pt')
predict_dir = os.path.join(project_root, 'runs/detect/predict')

# Load the best trained model
best_model = YOLO(best_model_path)

# Run predictions on validation images and save results
best_model.predict(source=val_img_path, save=True, project=os.path.join(project_root, 'runs/detect'), name='predict')

# Collect results (support both .jpg and .jpeg)
result_images = sorted(glob.glob(f'{predict_dir}/*.jpg') +
                        glob.glob(f'{predict_dir}/*.jpeg'))[:5]

if not result_images:
    print('No result images found. Check that prediction ran successfully.')
else:
    fig, axes = plt.subplots(1, len(result_images), figsize=(20, 5))
    if len(result_images) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, result_images):
        ax.imshow(mpimg.imread(img_path))
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## 7. Evaluate Accuracy

Key metrics for object detection:
- **mAP50** — mean Average Precision at IoU threshold 0.50 (main metric to watch)
- **mAP50-95** — stricter average across IoU thresholds 0.50–0.95
- **Precision** — of all boxes predicted, how many were correct
- **Recall** — of all real objects, how many were found

In [ ]:
# Run validation on the validation set
metrics = best_model.val(data=path_to_data_yaml)

# Overall metrics
print('=== Overall ===')
print(f'mAP50:      {metrics.box.map50:.3f}')
print(f'mAP50-95:   {metrics.box.map:.3f}')
print(f'Precision:  {metrics.box.mp:.3f}')
print(f'Recall:     {metrics.box.mr:.3f}')

# Per-class metrics
print('\n=== Per Class ===')
for i, cls_name in enumerate(classes):
    print(f'{cls_name:10s}  mAP50: {metrics.box.ap50[i]:.3f}  Precision: {metrics.box.p[i]:.3f}  Recall: {metrics.box.r[i]:.3f}')
